# 第 51 章 · 视觉接到大语言模型：图片变成"外语单词"，插进文字序列

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/51-vision-llm.html) · 仓库里 `beyond-transformer/notes/lessons/51-vision-llm.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 LLaVA 投影 | $H_v = Z_v W$ | `Z @ W` | [[2, 1], [0.5, 1.5]] |
| 第二步 拼进序列 | 拼接 | `build_sequence(H_v, text_emb)` | 图片 token 在前、文字 token 在后 |
| 第三步 BLIP-2 压缩 | 个数 | `257 * 1024`、`32 * 768` | 263168 → 24576 |
| 第四步 Qwen2-VL token 数 | (H/14)(W/14)/4 + 2 | `qwen2vl_tokens(H, W)` | 224² → 66；448² → 258；448 × 224 → 130 |
| 第五步 M-RoPE | 三个编号 | `mrope_ids([...])` | 文字 (0,0,0)…；图 (3,3,3)–(3,4,4)；"。" (5,5,5) |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
Z = np.array([[1, 0, 2], [0, 1, 1.0]])  # 两个小块的图片特征（编的，3 维）
W = np.array([[1, 0], [0, 1], [0.5, 0.5]])  # 投影：3 维 → 词向量 2 维

**函数 `build_sequence`**：图片 token 排在文字 token 前面，拼成一条序列。

In [ ]:
def build_sequence(img_tokens, text_emb):
    """图片 token 排在文字 token 前面，拼成一条序列。"""
    return np.concatenate([img_tokens, text_emb])

**函数 `qwen2vl_tokens`**：Qwen2-VL：按原尺寸切 14 × 14，相邻 2 × 2 合并成 1 个，再加首尾两个特殊 token。

In [ ]:
def qwen2vl_tokens(H, W, patch=14, merge=2):
    """Qwen2-VL：按原尺寸切 14 × 14，相邻 2 × 2 合并成 1 个，再加首尾两个特殊 token。"""
    n = (H // patch) * (W // patch)
    return n // (merge * merge) + 2

**函数 `mrope_ids`**：M-RoPE 编号。segments 是列表：("text", 个数) 或 ("image", 行, 列)。返回每个 token 的 (时间, 高, 宽)。

In [ ]:
def mrope_ids(segments):
    """M-RoPE 编号。segments 是列表：("text", 个数) 或 ("image", 行, 列)。返回每个 token 的 (时间, 高, 宽)。"""
    ids, start = [], 0
    for seg in segments:
        if seg[0] == "text":
            for i in range(seg[1]):
                ids.append((start + i,) * 3)
        else:
            _, rows, cols = seg
            for r in range(rows):
                for c in range(cols):
                    ids.append((start, start + r, start + c))
        start = max(max(i) for i in ids) + 1  # 下一段从最大编号 + 1 开始
    return ids

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 LLaVA：H_v = Z_v · W")
H_v = Z @ W
print(H_v)
text = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]])  # "这张 图 里" 的词向量（编的）
print("拼成序列（前 2 行是图片 token）：\n", build_sequence(H_v, text))

In [ ]:
print("\n第三步 BLIP-2：257 × 1024 =", 257 * 1024, "→ 32 × 768 =", 32 * 768)

In [ ]:
print("\n第四步 Qwen2-VL token 数")
for h, w in [(224, 224), (448, 448), (448, 224), (1344, 896)]:
    print(f"{h} × {w} →", qwen2vl_tokens(h, w))

In [ ]:
print("\n第五步 M-RoPE：文字 3 个 + 图片 2 × 2 + 文字 1 个")
for tok, i in zip(["描述", "这张", "图", "左上", "右上", "左下", "右下", "。"],
                  mrope_ids([("text", 3), ("image", 2, 2), ("text", 1)])):
    print(tok, i)

## 3. 代码里才看得到的

- 原文写 $W \cdot Z_v$（列向量），代码按行存特征，写成 `Z @ W`；W 的形状要跟着转置，这和前作第 12 章"存的是转置"是同一个坑。
- Qwen2-VL 要求宽高都能被 28（= 14 × 2）整除，实际实现会先把图片缩放到最接近的 28 的倍数；讲义里的例子都挑了能整除的尺寸。

## 4. 自己改一改

- `qwen2vl_tokens(896, 896)`、`qwen2vl_tokens(1792, 1792)`：看边长翻倍 token 涨几倍。
- `mrope_ids` 里把图片换成 3 帧的视频（时间编号每帧加 1），看编号怎么排。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 作家只读自家词，图片先请翻译来：编码器冻住，连接器来训，长度对齐插队前。
> LLaVA 一个矩阵投，先对齐、再微调。
> BLIP-2 派记者，三十二个去采访，交叉注意力带摘要。
> Qwen 不缩原图切，二乘二合一，首尾加标记。
> 位置拆成三个号：时间高宽，文字三同；换段从最大加一接着编。